In [3]:
!pip install -U "transformers>=4.46.2" "peft==0.17.0" "accelerate>=0.34.0" "bitsandbytes>=0.43.0" "safetensors>=0.4.5"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 124.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 991.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
from huggingface_hub import login
login()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments, Trainer, default_data_collator
from datasets import Dataset
import os
import glob
import json

In [ ]:
checkpoint = "meta-llama/Meta-Llama-3.1-8B-Instruct"
device = "cuda"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [ ]:
paths = sorted(glob.glob("edu_chat_nav_dataset.large.part*.jsonl"))
assert paths, "Put your JSONL files in Files tab"

max_len = 1024

def iter_examples(paths):
    for p in paths:
        with open(p, "r", encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                conv = obj.get("conversations", [])
                msgs = [{"role": c["from"], "content": c["value"]} for c in conv]
                for i, m in enumerate(msgs):
                    if m["role"] != "assistant":
                        continue
                    ctx = msgs[:i]
                    prompt_ids = tokenizer.apply_chat_template(
                        ctx,
                        tokenize=True,
                        add_generation_prompt=True,
                        return_tensors=None,
                    )
                    ans_ids = tokenizer(m["content"], add_special_tokens=False)["input_ids"]
                    if not ans_ids or ans_ids[-1] != tokenizer.eos_token_id:
                        ans_ids = ans_ids + [tokenizer.eos_token_id]
                    input_ids = prompt_ids + ans_ids
                    if len(input_ids) > max_len:
                        input_ids = input_ids[-max_len:]
                    cut = max(0, len(input_ids) - len(ans_ids))
                    labels = [-100]*cut + input_ids[cut:]
                    yield {
                        "input_ids": input_ids,
                        "labels": labels,
                        "attention_mask": [1]*len(input_ids),
                    }

samples = list(iter_examples(paths))
ds = Dataset.from_list(samples).train_test_split(test_size=0.05, seed=42)
train_dataset, eval_dataset = ds["train"], ds["test"]

In [ ]:
train_dataset, eval_dataset

(Dataset({
     features: ['input_ids', 'labels', 'attention_mask'],
     num_rows: 7253
 }),
 Dataset({
     features: ['input_ids', 'labels', 'attention_mask'],
     num_rows: 382
 }))

In [ ]:
bnb = BitsAndBytesConfig(
    load_in_4_bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    checkpoint,
    quantization_config=bnb,
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation="sdpa"
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [ ]:
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

In [ ]:
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

In [ ]:
model = get_peft_model(model, lora)

In [ ]:
args = TrainingArguments(
    output_dir="/content/out",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    num_train_epochs=1,
    fp16=True,
    max_grad_norm=1.0,
    optim="paged_adamw_8bit",
    logging_steps=50,
    save_strategy="epoch",
    report_to="none",
)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=default_data_collator
)

In [ ]:
trainer.train()

Step,Training Loss
50,0.335400
100,0.015500
150,0.004700
200,0.005200
250,0.004500
300,0.003300
350,0.004500
400,0.004100
450,0.003600


TrainOutput(global_step=454, training_loss=0.04198019272578051, metrics={'train_runtime': 6625.0838, 'train_samples_per_second': 1.095, 'train_steps_per_second': 0.069, 'total_flos': 3.239625790247731e+16, 'train_loss': 0.04198019272578051, 'epoch': 1.0})

In [ ]:
from huggingface_hub import create_repo, upload_folder
import shutil

In [ ]:
OUT_DIR = "/content/out"
EXPORT_DIR = "/content/hf_adapter"

In [ ]:
os.makedirs(EXPORT_DIR, exist_ok=True)

needed = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "tokenizer.json",
    "special_tokens_map.json",
    "chat_template.jinja",
]
for f in needed:
    src = os.path.join(OUT_DIR, f)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(EXPORT_DIR, f))

In [ ]:
repo_id = "ryanzhangofficial/synthellama"

In [ ]:
create_repo(repo_id, private=True, exist_ok=True)

upload_folder(
    repo_id=repo_id,
    folder_path=EXPORT_DIR,
    repo_type="model",
)
print("Pushed to:", f"https://huggingface.co/{repo_id}")

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...f_adapter/adapter_model.safetensors:   0%|          |  556kB /  168MB            

  /content/hf_adapter/tokenizer.json    : 100%|##########| 17.2MB / 17.2MB            

Pushed to: https://huggingface.co/RZJournal/synthellama


In [1]:
# Use the SyntheLlama Model
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE = "meta-llama/Meta-Llama-3.1-8B-Instruct"
ADAPTER = "ryanzhangofficial/synthellama"

tokenizer = AutoTokenizer.from_pretrained(BASE, use_fast=True)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=bnb,
    device_map="auto",
)
model = PeftModel.from_pretrained(model, ADAPTER)
model.eval()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/949 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [ ]:
while True:
    user_question = input("Enter your question (or 'quit' to exit): ")
    if user_question.lower() in {"quit", "exit"}:
        break

    msgs = [
        {"role": "system", "content": "You are the Synthesis Tutorial Copilot. Ask before running actions."},
        {"role": "user", "content": user_question}
    ]

    prompt = tokenizer.apply_chat_template(
        msgs,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
        )

    print(tokenizer.decode(out[0], skip_special_tokens=True))